# 04 Model validation
Absorbing chain, rolling-origin backtest against baselines, and the first-order tests.

In [1]:
import logging, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.getLogger("streamlit").setLevel(logging.ERROR)
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)

In [2]:
from src.services import rolls, forecast
ab = rolls.absorbing_table('with_dormant')
ab[['STATE', 'p_co', 'p_co_lo', 'p_co_hi', 'p_paid', 'months', 'months_lo', 'months_hi']].round(3)

2026-10-04 20:32:38.743 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-04 20:32:38.746 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-04 20:32:38.749 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


,STATE,p_co,p_co_lo,p_co_hi,p_paid,months,months_lo,months_hi
0,DORMANT,0.014,0.011,0.017,0.986,22.630,22.287,22.969
1,1-29,0.012,0.010,0.014,0.988,21.051,20.742,21.349
2,30-59,0.026,0.022,0.030,0.974,19.292,18.938,19.619
3,60-89,0.202,0.170,0.235,0.798,15.455,14.624,16.270
4,90-119,0.514,0.454,0.575,0.486,8.703,7.568,9.913
5,120-149,0.749,0.687,0.809,0.251,3.543,2.642,4.576


In [3]:
rolls.point_of_no_return()

2026-10-04 20:32:38.790 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


{'with_dormant': {'bucket': '90-119',
  'replicate_share_same': 0.6765,
  'replicate_distribution': {'90-119': 0.6765, '120-149': 0.3235}},
 'dormant_dropped': {'bucket': '90-119',
  'replicate_share_same': 0.983,
  'replicate_distribution': {'90-119': 0.983, '120-149': 0.017}}}

## Backtest: mean absolute state-share error (pp) by model and horizon

In [4]:
b = forecast.backtest()
m = b[(b.metric == 'mae_pp') & b.model.isin(forecast.MODEL_LABELS)]
m.groupby(['horizon', 'model']).value.mean().unstack().round(2)

model,markov,markov_rolling,mob_conditioned,oracle,persistence,static_mix
horizon,,,,,,
3,0.77,0.62,0.52,0.59,4.01,8.95
6,1.37,1.17,0.73,1.17,7.35,5.47
12,1.81,1.66,1.18,1.70,11.87,2.51


In [5]:
forecast.skill(12).query("metric == 'mae_pp'")[['origin', 'model', 'value', 'lo', 'hi']].round(3)

,origin,model,value,lo,hi
348,-36,skill_vs_persistence,0.819,0.795,0.843
355,-36,skill_vs_static_mix,0.441,0.395,0.491
726,-33,skill_vs_persistence,0.858,0.832,0.884
733,-33,skill_vs_static_mix,0.402,0.337,0.465
1104,-30,skill_vs_persistence,0.869,0.844,0.890
1111,-30,skill_vs_static_mix,0.223,0.133,0.294
1482,-27,skill_vs_persistence,0.858,0.834,0.877
1489,-27,skill_vs_static_mix,-0.054,-0.128,0.080
1860,-24,skill_vs_persistence,0.884,0.860,0.895
1867,-24,skill_vs_static_mix,0.130,-0.057,0.202


## Charge-off count calibration

In [6]:
cal = forecast.calibration()
print(int(cal.inside.sum()), 'of', len(cal), 'actual counts inside the 95% predictive interval')
cal.round(1).tail(9)

21 of 32 actual counts inside the 95% predictive interval


,origin,horizon,cohort_n,observed_n,missing_n,predicted_co,actual_co,pred_lo,pred_hi,inside
23,-15,12,10701.0,8896.0,1805.0,48.4,24.0,33.0,65.0,False
24,-12,3,11336.0,10771.0,565.0,14.0,11.0,6.0,23.0,True
25,-12,6,11336.0,10285.0,1051.0,28.8,16.0,18.0,42.0,False
26,-12,12,11336.0,9263.0,2073.0,50.0,31.0,34.0,68.0,False
27,-9,3,11881.0,11369.0,512.0,15.7,10.0,7.0,26.0,True
28,-9,6,11881.0,10858.0,1023.0,31.4,22.0,19.0,46.0,True
29,-6,3,12543.0,12005.0,538.0,18.7,16.0,9.0,30.0,True
30,-6,6,12543.0,11339.0,1204.0,34.7,27.0,22.0,49.0,True
31,-3,3,12969.0,12255.0,714.0,17.9,20.0,9.0,28.0,True


## Is first-order Markov enough?

In [7]:
st, summary = forecast.stationarity()
mh, mb, ht = forecast.mob_tests()
print(summary)
mh.round(4)

{'rows_rejecting_at_5pct': 4, 'rows_tested': 6, 'oot_mae_pp_origin_m12_h12': {'first_half': 3.4546749533007812, 'expanding_to_m12': 2.992693814372891}}


,FROM_STATE,chi2,dof,p_value,n
0,DORMANT,1886.5241,14,0.0000,55166.0
1,1-29,1960.2216,14,0.0000,372573.0
2,30-59,83.1814,12,0.0000,10737.0
3,60-89,57.6779,12,0.0000,849.0
4,90-119,42.0705,14,0.0001,314.0
5,120-149,51.9207,12,0.0000,218.0


In [8]:
ht.round(4)

,CUR_STATE,chi2,dof,p_value,n
0,30-59,987.1892,24,0.0,10549.0
1,60-89,252.4321,30,0.0,837.0
2,90-119,190.1003,35,0.0,308.0
3,120-149,121.2512,24,0.0,213.0
